# Group Lab - Build a regression model in 10 minutes
### Topic 9  |  4 groups  |  4 models  |  real garment factory data

| Group | Model | You write |
|---|---|---|
| **1** | Linear regression | gradient + prediction |
| **2** | Ridge regression (L2) | gradient + prediction |
| **3** | Lasso regression (L1) | gradient + prediction |
| **4** | Logistic regression | gradient + probability + prediction |

**Rules (same as AutoCode)**
1. **Training is already written** in the base class `AnalyticalModel`: it runs gradient descent for you. You only write the **gradient** of the loss and the **prediction**.
2. Your model is tested on the **real garment dataset** and must **beat the baseline**. [Dataset info](https://archive.ics.uci.edu/dataset/597/productivity+prediction+of+garment+employees)

**How to work (10 minutes)**
1. **File -> Save a copy in Drive**, run **SETUP** (upload `garments_worker_productivity.csv` when asked).
2. Go to **your group's section only**. Replace every `...` (look for **TODO**).
3. Run the **CHECK** cell until everything is ✅ green.
4. Run the **PRESENTATION CARD** cell and share your screen when you present.

**Roles:** Driver (types)  |  Navigator (reads the formula)  |  Checker (runs checks)  |  Presenter (presents the card)

In [ ]:
# SETUP - run once (do not change)
import os, inspect
import numpy as np, pandas as pd
from IPython.display import HTML, display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FILE = "garments_worker_productivity.csv"
if not os.path.exists(FILE):
    from google.colab import files
    print("Upload garments_worker_productivity.csv"); files.upload()
df = pd.read_csv(FILE); df["wip"] = df["wip"].fillna(0)
FEATURES = ["targeted_productivity","smv","wip","over_time","incentive","idle_time","idle_men","no_of_style_change","no_of_workers"]
X, y = df[FEATURES].values, df["actual_productivity"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)     # scaled (important for Ridge/Lasso!)
yc = (y >= 0.75).astype(int)                                                          # classification: HIGH productivity?
Xctr, Xcte, yctr, ycte = train_test_split(X, yc, test_size=0.3, random_state=0, stratify=yc)
scc = StandardScaler().fit(Xctr); Xctr, Xcte = scc.transform(Xctr), scc.transform(Xcte)
BASE_MSE = np.mean((yte - ytr.mean())**2)
BASE_ACC = max(ycte.mean(), 1 - ycte.mean())

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

class AnalyticalModel:
    """Like tasks/base.py: training with gradient descent is DONE for you."""
    def __init__(self, lr=0.1, n_iter=2000, lam=0.0):
        self.lr, self.n_iter, self.lam = lr, n_iter, lam
    def add_bias(self, X):
        return np.c_[np.ones(len(X)), X]            # column of 1s for the intercept b0
    def fit(self, X, y):
        Xb = self.add_bias(X)
        self.w = np.zeros(Xb.shape[1])
        for _ in range(self.n_iter):
            self.w = self.w - self.lr * self._gradient(Xb, y)   # <- calls YOUR gradient
        return self
    def _gradient(self, X, y):
        raise NotImplementedError("write the gradient")
    def predict(self, X):
        raise NotImplementedError("write the prediction")

# ---- checking helpers ----
RESULTS = {}
def show(group, name, ok, hint):
    RESULTS[(group, name)] = ok
    col, bg, icon = ("#2e7d32","#e8f5e9","✅") if ok else ("#c62828","#ffebee","❌")
    display(HTML(f'<div style="background:{bg};border-left:8px solid {col};padding:9px 14px;margin:4px 0;font-family:Arial;font-size:15px;color:#222">'
                 f'<b style="color:{col}">{icon} {name}</b> - {"Correct!" if ok else "Not yet. Hint: " + hint}</div>'))
def safe(f):
    try: return bool(f())
    except Exception: return False
def numeric_grad(loss, w, h=1e-6):                 # the Topic 7 slope formula
    g = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        g[i] = (loss(w + e) - loss(w - e)) / (2*h)
    return g
def grad_ok(model, Xb, yy, loss):
    model.w = np.random.default_rng(1).normal(0, 0.5, Xb.shape[1])
    return np.allclose(model._gradient(Xb, yy), numeric_grad(loss, model.w.copy()), atol=1e-4)
rng = np.random.default_rng(0); Xs = np.c_[np.ones(20), rng.normal(size=(20, 3))]; ys = rng.normal(size=20); ysc = (ys > 0).astype(float)
print(f"Setup done. Baseline MSE = {BASE_MSE:.4f} | baseline accuracy = {BASE_ACC:.3f}")

---
# GROUP 1 - Linear regression
**Prediction:** $\hat{y} = Xw$  **Loss:** MSE $= \frac{1}{n}\sum(Xw - y)^2$  **Gradient:** $\frac{2}{n} X^T (Xw - y)$

In NumPy: `X.T @ error` is $X^T \cdot$ error, and `Xb @ self.w` is $Xw$.

In [ ]:
class MyLinearRegression(AnalyticalModel):
    def _gradient(self, X, y):          # X already has the column of 1s; self.w = current weights
        n = len(y)
        error = X @ self.w - y
        grad = ...                      # TODO: (2 / n) times X transposed times error
        return grad

    def predict(self, X):
        Xb = self.add_bias(X)
        return ...                      # TODO: Xb times the weights

In [ ]:
# CHECK - Group 1
show(1, "gradient matches the true slope", safe(lambda: grad_ok(MyLinearRegression(), Xs, ys, lambda w: np.mean((Xs @ w - ys)**2))), "grad = (2 / n) * X.T @ error")
show(1, "learns y = 2x (predicts 8 for x = 4)", safe(lambda: np.isclose(MyLinearRegression(lr=0.05).fit(np.array([[1.],[2],[3]]), np.array([2.,4,6])).predict(np.array([[4.]]))[0], 8, atol=0.05)), "predict: return Xb @ self.w")
m1 = None
def _t1():
    global m1; m1 = MyLinearRegression().fit(Xtr, ytr); return np.mean((m1.predict(Xte) - yte)**2) < BASE_MSE
show(1, "beats the baseline on garment data", safe(_t1), "both TODOs must be correct")

In [ ]:
# PRESENTATION CARD - Group 1
try:
    src = inspect.getsource(MyLinearRegression._gradient).replace("<","&lt;")
    mse = np.mean((m1.predict(Xte) - yte)**2); result = f"test MSE {mse:.4f} vs baseline {BASE_MSE:.4f} ({1-mse/BASE_MSE:.0%} better)"
except Exception as e:
    src, result = "(finish the TODOs first)", "not ready yet"
passed = [ok for (grp, _), ok in RESULTS.items() if grp == 1]
status = "✅ ALL CHECKS PASSED" if passed and all(passed) else f"{sum(passed)}/{len(passed)} checks passed"
display(HTML(f"""<div style="font-family:Arial;border:3px solid #39C2D7;border-radius:12px;padding:18px 22px;max-width:820px">
<div style="background:#39C2D7;color:white;font-weight:bold;font-size:20px;padding:8px 14px;border-radius:8px">Group 1: Linear regression</div>
<p style="font-size:16px"><b>1. Our formula:</b> y&#770; = X w,  gradient = (2/n) X<sup>T</sup>(Xw - y)</p>
<p style="font-size:16px"><b>2. Our code for the gradient:</b></p><pre style="background:#263852;color:#fff;padding:12px;border-radius:8px;font-size:13px">{src}</pre>
<p style="font-size:16px"><b>3. Our result on the garment data:</b> {result}</p>
<p style="font-size:15px;color:#263852"><b>{status}</b></p></div>"""))

---
# GROUP 2 - Ridge regression (L2)
**Loss:** MSE $+ \lambda \sum w^2$  **Gradient:** $\frac{2}{n}X^T(Xw-y) + 2\lambda w$  (do **not** penalise the intercept $w_0$)

The linear part is already written for you - add the penalty. `self.lam` is $\lambda$.

In [ ]:
class MyRidge(AnalyticalModel):
    def _gradient(self, X, y):
        n = len(y)
        error = X @ self.w - y
        grad = (2 / n) * X.T @ error    # the linear part (given)
        penalty = ...                   # TODO: 2 times lambda times the weights
        penalty[0] = 0                  # do not penalise the intercept
        return grad + penalty

    def predict(self, X):
        Xb = self.add_bias(X)
        return ...                      # TODO: Xb times the weights

In [ ]:
# CHECK - Group 2
show(2, "gradient matches the true slope", safe(lambda: grad_ok(MyRidge(lam=0.3), Xs, ys, lambda w: np.mean((Xs @ w - ys)**2) + 0.3*np.sum(w[1:]**2))), "penalty = 2 * self.lam * self.w")
def _shrink():
    a = MyRidge(lam=0.0).fit(Xtr, ytr).w[1:]; b = MyRidge(lam=1.0).fit(Xtr, ytr).w[1:]
    return np.sum(b**2) < 0.8 * np.sum(a**2)
show(2, "bigger lambda -> smaller weights", safe(_shrink), "the penalty must use self.lam")
m2 = None
def _t2():
    global m2; m2 = MyRidge(lam=0.1).fit(Xtr, ytr); return np.mean((m2.predict(Xte) - yte)**2) < BASE_MSE
show(2, "beats the baseline on garment data", safe(_t2), "predict: return Xb @ self.w")

In [ ]:
# PRESENTATION CARD - Group 2
try:
    src = inspect.getsource(MyRidge._gradient).replace("<","&lt;")
    mse = np.mean((m2.predict(Xte) - yte)**2); w_smv, w_wk = m2.w[2], m2.w[9]; result = f"test MSE {mse:.4f} vs baseline {BASE_MSE:.4f}; twins smv {w_smv:+.3f} / workers {w_wk:+.3f}"
except Exception as e:
    src, result = "(finish the TODOs first)", "not ready yet"
passed = [ok for (grp, _), ok in RESULTS.items() if grp == 2]
status = "✅ ALL CHECKS PASSED" if passed and all(passed) else f"{sum(passed)}/{len(passed)} checks passed"
display(HTML(f"""<div style="font-family:Arial;border:3px solid #39C2D7;border-radius:12px;padding:18px 22px;max-width:820px">
<div style="background:#39C2D7;color:white;font-weight:bold;font-size:20px;padding:8px 14px;border-radius:8px">Group 2: Ridge regression (L2)</div>
<p style="font-size:16px"><b>1. Our formula:</b> loss = MSE + &lambda; &Sigma;w&sup2;,  gradient = linear + 2&lambda;w</p>
<p style="font-size:16px"><b>2. Our code for the gradient:</b></p><pre style="background:#263852;color:#fff;padding:12px;border-radius:8px;font-size:13px">{src}</pre>
<p style="font-size:16px"><b>3. Our result on the garment data:</b> {result}</p>
<p style="font-size:15px;color:#263852"><b>{status}</b></p></div>"""))

---
# GROUP 3 - Lasso regression (L1)
**Loss:** MSE $+ \lambda \sum |w|$  **Gradient:** $\frac{2}{n}X^T(Xw-y) + \lambda\,\mathrm{sign}(w)$  (do **not** penalise $w_0$)

$|w|$ has no derivative at 0, so we use `np.sign(w)` = -1, 0 or +1 (a *subgradient*).

In [ ]:
class MyLasso(AnalyticalModel):
    def _gradient(self, X, y):
        n = len(y)
        error = X @ self.w - y
        grad = (2 / n) * X.T @ error    # the linear part (given)
        penalty = ...                   # TODO: lambda times the SIGN of the weights (np.sign)
        penalty[0] = 0                  # do not penalise the intercept
        return grad + penalty

    def predict(self, X):
        Xb = self.add_bias(X)
        return ...                      # TODO: Xb times the weights

In [ ]:
# CHECK - Group 3
show(3, "gradient matches the true slope", safe(lambda: grad_ok(MyLasso(lam=0.3), Xs, ys, lambda w: np.mean((Xs @ w - ys)**2) + 0.3*np.sum(np.abs(w[1:])))), "penalty = self.lam * np.sign(self.w)")
show(3, "big lambda pushes several weights to ~0", safe(lambda: np.sum(np.abs(MyLasso(lam=0.03).fit(Xtr, ytr).w[1:]) < 0.01) >= 4), "use np.sign(self.w), not self.w")
m3 = None
def _t3():
    global m3; m3 = MyLasso(lam=0.01).fit(Xtr, ytr); return np.mean((m3.predict(Xte) - yte)**2) < BASE_MSE
show(3, "beats the baseline on garment data", safe(_t3), "predict: return Xb @ self.w")

In [ ]:
# PRESENTATION CARD - Group 3
try:
    src = inspect.getsource(MyLasso._gradient).replace("<","&lt;")
    mse = np.mean((m3.predict(Xte) - yte)**2); near0 = [f for f, v in zip(FEATURES, m3.w[1:]) if abs(v) < 0.005]; result = f"test MSE {mse:.4f} vs baseline {BASE_MSE:.4f}; features pushed to ~0: {near0}"
except Exception as e:
    src, result = "(finish the TODOs first)", "not ready yet"
passed = [ok for (grp, _), ok in RESULTS.items() if grp == 3]
status = "✅ ALL CHECKS PASSED" if passed and all(passed) else f"{sum(passed)}/{len(passed)} checks passed"
display(HTML(f"""<div style="font-family:Arial;border:3px solid #39C2D7;border-radius:12px;padding:18px 22px;max-width:820px">
<div style="background:#39C2D7;color:white;font-weight:bold;font-size:20px;padding:8px 14px;border-radius:8px">Group 3: Lasso regression (L1)</div>
<p style="font-size:16px"><b>1. Our formula:</b> loss = MSE + &lambda; &Sigma;|w|,  gradient = linear + &lambda;&middot;sign(w)</p>
<p style="font-size:16px"><b>2. Our code for the gradient:</b></p><pre style="background:#263852;color:#fff;padding:12px;border-radius:8px;font-size:13px">{src}</pre>
<p style="font-size:16px"><b>3. Our result on the garment data:</b> {result}</p>
<p style="font-size:15px;color:#263852"><b>{status}</b></p></div>"""))

---
# GROUP 4 - Logistic regression (classification)
Question: **is this team-day HIGH productivity (actual >= 0.75)?**
**Probability:** $p = \sigma(Xw)$  **Loss:** log-loss  **Gradient:** $\frac{1}{n}X^T(p - y)$  **Class:** 1 if $p \ge 0.5$

`sigmoid(z)` is already defined in SETUP.

In [ ]:
class MyLogistic(AnalyticalModel):
    def _gradient(self, X, y):
        n = len(y)
        p = ...                         # TODO: sigmoid of (X times the weights)
        return ...                      # TODO: X transposed times (p - y), divided by n

    def predict_proba(self, X):
        Xb = self.add_bias(X)
        return ...                      # TODO: sigmoid of (Xb times the weights)

    def predict(self, X, threshold=0.5):
        return ...                      # TODO: 1 if probability >= threshold else 0  (hint: (...).astype(int))

In [ ]:
# CHECK - Group 4
def _ll(w):
    p = np.clip(sigmoid(Xs @ w), 1e-12, 1 - 1e-12); return -np.mean(ysc*np.log(p) + (1 - ysc)*np.log(1 - p))
show(4, "gradient matches the true slope", safe(lambda: grad_ok(MyLogistic(), Xs, ysc, _ll)), "p = sigmoid(X @ self.w); return X.T @ (p - y) / n")
m4 = None
def _probs():
    global m4; m4 = MyLogistic(lr=0.5).fit(Xctr, yctr); pr = m4.predict_proba(Xcte); return pr.min() >= 0 and pr.max() <= 1 and pr.std() > 0.05
show(4, "probabilities are between 0 and 1", safe(_probs), "predict_proba: return sigmoid(Xb @ self.w)")
show(4, "beats the baseline accuracy on garment data", safe(lambda: np.mean(m4.predict(Xcte) == ycte) > BASE_ACC), "predict: (self.predict_proba(X) >= threshold).astype(int)")

In [ ]:
# PRESENTATION CARD - Group 4
try:
    src = inspect.getsource(MyLogistic._gradient).replace("<","&lt;")
    acc = np.mean(m4.predict(Xcte) == ycte); result = f"accuracy {acc:.1%} vs baseline {BASE_ACC:.1%}"
except Exception as e:
    src, result = "(finish the TODOs first)", "not ready yet"
passed = [ok for (grp, _), ok in RESULTS.items() if grp == 4]
status = "✅ ALL CHECKS PASSED" if passed and all(passed) else f"{sum(passed)}/{len(passed)} checks passed"
display(HTML(f"""<div style="font-family:Arial;border:3px solid #39C2D7;border-radius:12px;padding:18px 22px;max-width:820px">
<div style="background:#39C2D7;color:white;font-weight:bold;font-size:20px;padding:8px 14px;border-radius:8px">Group 4: Logistic regression</div>
<p style="font-size:16px"><b>1. Our formula:</b> p = &sigma;(Xw),  gradient = (1/n) X<sup>T</sup>(p - y),  class = p &ge; 0.5</p>
<p style="font-size:16px"><b>2. Our code for the gradient:</b></p><pre style="background:#263852;color:#fff;padding:12px;border-radius:8px;font-size:13px">{src}</pre>
<p style="font-size:16px"><b>3. Our result on the garment data:</b> {result}</p>
<p style="font-size:15px;color:#263852"><b>{status}</b></p></div>"""))

---
# After presenting: what did we learn?
- Groups 1, 2 and 3 wrote **almost the same code**: Ridge and Lasso are linear regression **plus one penalty line**.
- Group 4's gradient has the **same shape** as Group 1's: $X^T(\text{prediction} - \text{truth})$ - only the sigmoid is new.
- In **AutoCode** you will do **all four on your own**. Follow the conventions in `tasks/base.py` - names and scaling may differ slightly from this lab.